In [ ]:
# Project-root setup (auto-injected during notebook reorg 2026-05-06).
# Walks up from the notebook's working directory until it finds pyproject.toml,
# then adds that directory to sys.path so `from twopoint_lockin import ...` etc. work
# regardless of which subfolder the notebook lives in. PROJECT_ROOT is also exposed
# so cells below can build absolute paths to data/ and the vendored qickdawg package.
import sys as _sys
from pathlib import Path as _Path
_p = _Path.cwd().resolve()
while _p != _p.parent and not (_p / "pyproject.toml").exists():
    _p = _p.parent
PROJECT_ROOT = _p if (_p / "pyproject.toml").exists() else _Path.cwd()
if str(PROJECT_ROOT) not in _sys.path:
    _sys.path.insert(0, str(PROJECT_ROOT))
# Also add notebook_modules/ so `from twopoint_lockin import ...` etc. work
# without per-notebook sys.path tweaks. The five .py modules used by these
# notebooks live there now (lockin_extensions, multipoint_lockin_program,
# nv_magnetometry_analysis, odmr_sensitivity, twopoint_lockin).
_nb_modules_dir = PROJECT_ROOT / "notebook_modules"
if _nb_modules_dir.exists() and str(_nb_modules_dir) not in _sys.path:
    _sys.path.insert(0, str(_nb_modules_dir))
print(f"PROJECT_ROOT: {PROJECT_ROOT}")


# Basic NV Center Testing — With Laser Noise Cancellation

**Enhanced version of `01_basic_nv_testing.ipynb`.**  
This notebook adds a second ADC readout channel for real-time laser intensity noise cancellation.

## Noise cancellation concept

Our NV PL signal contains two contributions:

$$S_{\text{ch1}}(t) = S_{\text{NV}}(t) + \eta_{\text{laser}}(t) + \eta_{\text{elec}}(t)$$

A reference detector on **ADC Channel 0** looks at the laser directly (no NV sample), capturing:

$$S_{\text{ch0}}(t) = \alpha\,\eta_{\text{laser}}(t) + \eta_{\text{elec,0}}(t)$$

The corrected (noise-cancelled) signal is:

$$S_{\text{corr}}(t) = S_{\text{ch1}}(t) - \frac{1}{\alpha}\,S_{\text{ch0}}(t) \approx S_{\text{NV}}(t)$$

The gain factor $\alpha$ is calibrated from the correlated noise floor (Section 3).

## Hardware setup
| Channel | Role |
|---------|------|
| ADC Channel 1 (C) | NV photoluminescence — photodiode on NV sample |
| ADC Channel 0 (A) | Laser noise reference — second photodiode on direct laser beam |

**Before running:** Laser manually ON, NV photodiode on Channel C (1), reference photodiode on Channel A (0).

## Sections
- Section 0 — Setup & Connection
- Section 1 — Default Configuration
- Section 2 — Dual-Channel PL Verification
- Section 3 — Noise Characterization & Gain Calibration
- Section 4 — Noise-Corrected PL Readout
- Section 5 — Noise-Corrected ODMR Spectrum
- Section 6 — Noise-Corrected Readout Window Calibration

---
## Section 0: Setup & Connection

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from copy import copy
import qickdawg as qd
from scipy.optimize import curve_fit
from scipy.signal import find_peaks as scipy_find_peaks

In [ ]:
# ===== EDIT THIS: Set your RFSoC IP address =====
RFSOC_IP = '172.16.26.5'

qd.start_client(RFSOC_IP)
print(f"Connected to RFSoC at {RFSOC_IP}")

---
## Section 1: Default Configuration

Two ADC channels are used:
- **`NV_CHANNEL = 1`** — Channel C, NV photoluminescence
- **`NOISE_CHANNEL = 0`** — Channel A, laser intensity noise reference

All measurement programs will be run once per channel and the noise correction applied afterwards.

In [ ]:
# ===== ADC channel assignments =====
NV_CHANNEL    = 1   # Channel C: NV PL signal
NOISE_CHANNEL = 0   # Channel A: laser noise reference

def make_config(adc_ch=NV_CHANNEL):
    """Return a fresh NVConfiguration with our fixed hardware settings."""
    cfg = qd.NVConfiguration()
    cfg.adc_channel     = adc_ch
    cfg.mw_channel      = 0
    cfg.mw_nqz          = 1
    cfg.mw_gain         = 5000
    cfg.laser_gate_pmod = 0        # Required by qickdawg; not wired to our laser
    cfg.relax_delay_tns = 500
    return cfg

default_config    = make_config(NV_CHANNEL)
default_config_n0 = make_config(NOISE_CHANNEL)

print("Default configurations ready.")
print(f"  NV channel:    {NV_CHANNEL}  (Channel C)")
print(f"  Noise channel: {NOISE_CHANNEL}  (Channel A)")

---
## Section 2: Dual-Channel PL Verification

Verify that both channels are responding to the laser:
- **Channel 1** should show the NV PL level (large positive value).
- **Channel 0** should show the laser reference level (large positive value).
- Both should drop near zero when the laser is blocked.

In [ ]:
# Single PL reading from each channel
cfg_pl_nv    = make_config(NV_CHANNEL)
cfg_pl_noise = make_config(NOISE_CHANNEL)

cfg_pl_nv.readout_integration_treg    = 2**16 - 1
cfg_pl_noise.readout_integration_treg = 2**16 - 1
cfg_pl_nv.reps    = 1
cfg_pl_noise.reps = 1

prog_pl_nv    = qd.PLIntensity(cfg_pl_nv)
prog_pl_noise = qd.PLIntensity(cfg_pl_noise)

val_nv    = prog_pl_nv.acquire()
val_noise = prog_pl_noise.acquire()

print(f"Channel {NV_CHANNEL} (NV PL):         {val_nv:.2f} ADC units")
print(f"Channel {NOISE_CHANNEL} (noise ref):    {val_noise:.2f} ADC units")
print()

for name, val in [(f'Ch{NV_CHANNEL} NV', val_nv), (f'Ch{NOISE_CHANNEL} noise', val_noise)]:
    if abs(val) < 1:
        print(f"WARNING: {name} is near zero. Check photodiode connection and laser.")
    else:
        print(f"OK: {name} signal detected.")

In [ ]:
# Laser block check:
# Run with laser ON → both channels should be large.
# Block laser → both should drop near zero.

val_nv    = prog_pl_nv.acquire()
val_noise = prog_pl_noise.acquire()

print(f"Ch{NV_CHANNEL} NV:    {val_nv:.2f}")
print(f"Ch{NOISE_CHANNEL} noise: {val_noise:.2f}")
print("Expected: both large with laser ON, both ~0 with laser blocked.")

In [ ]:
# Live dual-channel monitor — useful for verifying both detectors
# Shows: NV PL (ch1), noise ref (ch0), and their raw difference.
# Press Stop (■) or Ctrl+C to exit.

def get_dual_pl():
    v_nv    = prog_pl_nv.acquire(progress=False)
    v_noise = prog_pl_noise.acquire(progress=False)
    # Return as dict so live_plot can show labeled traces
    return {
        f'Ch{NV_CHANNEL} NV PL':         v_nv,
        f'Ch{NOISE_CHANNEL} noise ref':   v_noise,
        'Raw difference (ch1-ch0)':       v_nv - v_noise,
    }

# Uncomment to start:
# qd.live_plot(get_dual_pl)

---
## Section 3: Noise Characterization & Gain Calibration

### Goal
Find the gain factor **α** so that the corrected signal
$$S_{\text{corr}} = S_{\text{ch1}} - \alpha\,S_{\text{ch0}}$$
has minimum variance (maximum laser-noise rejection).

### Method
Collect N repeated PL readings from both channels.  
The optimal α is the OLS regression coefficient:
$$\alpha^* = \frac{\text{Cov}(S_{\text{ch1}},\, S_{\text{ch0}})}{\text{Var}(S_{\text{ch0}})}$$
This minimises $\text{Var}(S_{\text{ch1}} - \alpha\,S_{\text{ch0}})$.

> **Physical meaning:** α encodes the beam-splitting ratio and the relative responsivities of the two photodiodes.  
> A Pearson correlation $r$ close to ±1 means the two channels share a strong common noise source (laser intensity).

### When to re-run
Re-calibrate α whenever the optical path changes (realignment, neutral-density filters, etc.).

In [ ]:
# ===== EDIT: number of repeated samples for noise calibration =====
N_NOISE_SAMPLES = 200   # More → better α estimate; each sample takes ~1 readout cycle

cfg_ns_nv    = make_config(NV_CHANNEL)
cfg_ns_noise = make_config(NOISE_CHANNEL)

# Short integration so we sample fast enough to see laser fluctuations
cfg_ns_nv.readout_integration_treg    = 2**14
cfg_ns_noise.readout_integration_treg = 2**14
cfg_ns_nv.reps    = 1
cfg_ns_noise.reps = 1

prog_ns_nv    = qd.PLIntensity(cfg_ns_nv)
prog_ns_noise = qd.PLIntensity(cfg_ns_noise)

ch1_series = np.zeros(N_NOISE_SAMPLES)
ch0_series = np.zeros(N_NOISE_SAMPLES)

print(f"Collecting {N_NOISE_SAMPLES} samples from each channel...")
for i in range(N_NOISE_SAMPLES):
    ch1_series[i] = prog_ns_nv.acquire(progress=False)
    ch0_series[i] = prog_ns_noise.acquire(progress=False)
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{N_NOISE_SAMPLES}")

print("Done.")

In [ ]:
# ── Compute optimal gain factor α ─────────────────────────────────────────
alpha_opt = np.cov(ch1_series, ch0_series)[0, 1] / np.var(ch0_series)

corrected_series = ch1_series - alpha_opt * ch0_series

# Noise metrics
std_ch1  = np.std(ch1_series)
std_ch0  = np.std(ch0_series)
std_corr = np.std(corrected_series)
pearson_r = np.corrcoef(ch1_series, ch0_series)[0, 1]
noise_reduction_pct = (1 - std_corr / std_ch1) * 100

print("═══ Noise Calibration Results ═══")
print(f"  Optimal gain factor α:        {alpha_opt:.4f}")
print(f"  Pearson correlation r:         {pearson_r:+.4f}")
print(f"  Noise std — Ch{NV_CHANNEL}:          {std_ch1:.4f} ADC units")
print(f"  Noise std — Ch{NOISE_CHANNEL}:          {std_ch0:.4f} ADC units")
print(f"  Noise std — corrected:         {std_corr:.4f} ADC units")
print(f"  Noise reduction:               {noise_reduction_pct:.1f} %")
print()

if abs(pearson_r) < 0.3:
    print("WARNING: Low correlation (|r| < 0.3).")
    print("  The reference detector may not be seeing the same laser noise.")
    print("  Check the optical path to Channel 0 and detector alignment.")
elif abs(pearson_r) > 0.7:
    print("Good correlation — noise cancellation will be effective.")
else:
    print("Moderate correlation — partial noise cancellation expected.")

# Save α for use in later sections
ALPHA = alpha_opt
print(f"\nALPHA = {ALPHA:.4f}  (saved for all subsequent measurements)")

In [ ]:
# ── Plot noise time-series and power spectra ───────────────────────────────
sample_idx = np.arange(N_NOISE_SAMPLES)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Time series
ax = axes[0, 0]
ax.plot(sample_idx, ch1_series - np.mean(ch1_series), alpha=0.8, label=f'Ch{NV_CHANNEL} NV (mean sub.)', color='steelblue')
ax.plot(sample_idx, ch0_series * ALPHA - np.mean(ch0_series * ALPHA), alpha=0.8, label=f'α × Ch{NOISE_CHANNEL} noise ref', color='darkorange')
ax.set_xlabel('Sample index')
ax.set_ylabel('ADC units (mean-subtracted)')
ax.set_title('Raw channel fluctuations')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Corrected time series
ax = axes[0, 1]
ax.plot(sample_idx, ch1_series - np.mean(ch1_series),
        alpha=0.5, label='Before correction', color='steelblue')
ax.plot(sample_idx, corrected_series - np.mean(corrected_series),
        alpha=0.9, label='After correction', color='green')
ax.set_xlabel('Sample index')
ax.set_ylabel('ADC units (mean-subtracted)')
ax.set_title(f'Noise cancellation (reduction: {noise_reduction_pct:.1f} %)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Scatter: ch0 vs ch1 — should be linear if dominated by laser noise
ax = axes[1, 0]
ax.scatter(ch0_series, ch1_series, s=4, alpha=0.5, color='purple')
x_fit = np.linspace(ch0_series.min(), ch0_series.max(), 100)
ax.plot(x_fit, ALPHA * x_fit + (np.mean(ch1_series) - ALPHA * np.mean(ch0_series)),
        'r--', label=f'α = {ALPHA:.4f}')
ax.set_xlabel(f'Ch{NOISE_CHANNEL} noise reference (ADC units)')
ax.set_ylabel(f'Ch{NV_CHANNEL} NV PL (ADC units)')
ax.set_title(f'Correlation scatter  r = {pearson_r:+.3f}')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Power spectra (FFT)
ax = axes[1, 1]
freqs = np.fft.rfftfreq(N_NOISE_SAMPLES, d=1)
psd_ch1  = np.abs(np.fft.rfft(ch1_series - np.mean(ch1_series)))**2
psd_corr = np.abs(np.fft.rfft(corrected_series - np.mean(corrected_series)))**2
ax.semilogy(freqs[1:], psd_ch1[1:],  alpha=0.8, label=f'Ch{NV_CHANNEL} (before)', color='steelblue')
ax.semilogy(freqs[1:], psd_corr[1:], alpha=0.8, label='Corrected (after)', color='green')
ax.set_xlabel('Normalised frequency (1/sample)')
ax.set_ylabel('Power (ADC²)')
ax.set_title('Noise power spectrum')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.suptitle(f'Noise Characterisation  |  α = {ALPHA:.4f}  |  r = {pearson_r:+.3f}', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ===== OPTIONAL: manually override α =====
# Uncomment and set if you prefer a fixed value (e.g., from a previous calibration run)
# or if the auto-calibration gave an unreliable result.
#
# ALPHA = 1.0
# print(f"α overridden to: {ALPHA}")

print(f"Using α = {ALPHA:.4f} for all noise-corrected measurements below.")

---
## Section 4: Noise-Corrected PL Readout

Acquire PL intensity with and without noise correction.
The corrected value $S_{\text{corr}} = S_{\text{ch1}} - \alpha\,S_{\text{ch0}}$ directly tracks the NV spin population, with laser intensity fluctuations removed.

In [ ]:
def read_corrected_pl(alpha, n_avg=1):
    """Single noise-corrected PL reading.
    Returns (raw_ch1, raw_ch0, corrected) averaged over n_avg shots.
    """
    cfg_nv = make_config(NV_CHANNEL)
    cfg_n0 = make_config(NOISE_CHANNEL)
    cfg_nv.readout_integration_treg = 2**16 - 1
    cfg_n0.readout_integration_treg = 2**16 - 1
    cfg_nv.reps = n_avg
    cfg_n0.reps = n_avg

    v_nv = qd.PLIntensity(cfg_nv).acquire()
    v_n0 = qd.PLIntensity(cfg_n0).acquire()
    return v_nv, v_n0, v_nv - alpha * v_n0

v_nv, v_n0, v_corr = read_corrected_pl(ALPHA)

print("Noise-corrected PL readout:")
print(f"  Ch{NV_CHANNEL} NV PL (raw):          {v_nv:.2f} ADC units")
print(f"  Ch{NOISE_CHANNEL} noise ref (raw):    {v_n0:.2f} ADC units")
print(f"  α × Ch{NOISE_CHANNEL}:                {ALPHA * v_n0:.2f} ADC units")
print(f"  Corrected PL:                  {v_corr:.2f} ADC units")

In [ ]:
# Live noise-corrected PL monitor
# Useful for optimizing beam alignment — maximise corrected PL.
# Press Stop (■) or Ctrl+C to exit.

cfg_live_nv = make_config(NV_CHANNEL)
cfg_live_n0 = make_config(NOISE_CHANNEL)
cfg_live_nv.readout_integration_treg = 2**16 - 1
cfg_live_n0.readout_integration_treg = 2**16 - 1
cfg_live_nv.reps = 1
cfg_live_n0.reps = 1

prog_live_nv = qd.PLIntensity(cfg_live_nv)
prog_live_n0 = qd.PLIntensity(cfg_live_n0)

def get_corrected_pl():
    v_nv = prog_live_nv.acquire(progress=False)
    v_n0 = prog_live_n0.acquire(progress=False)
    return v_nv - ALPHA * v_n0

# Uncomment to start:
# qd.live_plot(get_corrected_pl)

---
## Section 5: Noise-Corrected ODMR Spectrum

**LockinODMR** is run on **both** ADC channels with identical sweep parameters.  
The noise-corrected contrast is:
$$\text{signal}_{\text{corr}} = \text{signal}_{\text{ch1}} - \alpha\,\text{signal}_{\text{ch0}}$$
$$\text{ref}_{\text{corr}} = \text{ref}_{\text{ch1}} - \alpha\,\text{ref}_{\text{ch0}}$$
$$\text{contrast}_{\text{corr}} = \frac{\text{ref}_{\text{corr}} - \text{signal}_{\text{corr}}}{\text{ref}_{\text{corr}}} \times 100\,\%$$

Because the MW on/off modulation is already a differential measurement, the noise cancellation primarily reduces the wideband noise floor, improving peak SNR.

In [ ]:
# Show the LockinODMR pulse sequence
qd.LockinODMR.plot_sequence()

In [ ]:
# ── ODMR configuration (shared between both channels) ──────────────────────
def make_odmr_config(adc_ch, mw_gain=10000, reps=10, start_MHz=2700, stop_MHz=3000, delta_MHz=1):
    cfg = make_config(adc_ch)
    cfg.readout_integration_tus = qd.max_int_time_tus
    cfg.mw_gain                 = mw_gain
    cfg.pre_init                = True
    cfg.reps                    = reps
    cfg.relax_delay_treg        = 500
    cfg.add_linear_sweep('mw', 'fMHz', start=start_MHz, stop=stop_MHz, delta=delta_MHz)
    return cfg

# ── Acquire ODMR on both channels ──────────────────────────────────────────
# Frequency sweep range — adjust to bracket your expected NV resonances
ODMR_START_MHZ = 2700
ODMR_STOP_MHZ  = 3000
ODMR_DELTA_MHZ = 1
ODMR_REPS      = 10

cfg_odmr_nv = make_odmr_config(
    NV_CHANNEL, reps=ODMR_REPS,
    start_MHz=ODMR_START_MHZ, stop_MHz=ODMR_STOP_MHZ, delta_MHz=ODMR_DELTA_MHZ
)
cfg_odmr_n0 = make_odmr_config(
    NOISE_CHANNEL, reps=ODMR_REPS,
    start_MHz=ODMR_START_MHZ, stop_MHz=ODMR_STOP_MHZ, delta_MHz=ODMR_DELTA_MHZ
)

prog_odmr_nv = qd.LockinODMR(cfg_odmr_nv)
prog_odmr_n0 = qd.LockinODMR(cfg_odmr_n0)

print(f"ODMR sweep: {ODMR_START_MHZ} → {ODMR_STOP_MHZ} MHz, Δ = {ODMR_DELTA_MHZ} MHz")
print(f"  Points: {cfg_odmr_nv.nsweep_points}  Reps: {ODMR_REPS}")
print(f"  Estimated time (NV channel): {prog_odmr_nv.total_time():.1f} s")
print()

print("[1/2] Acquiring ODMR on NV channel...")
d_nv = prog_odmr_nv.acquire(progress=True)
print("[2/2] Acquiring ODMR on noise channel...")
d_n0 = prog_odmr_n0.acquire(progress=True)
print("Done.")

In [ ]:
# ── Apply noise correction ─────────────────────────────────────────────────
signal_corr    = d_nv.signal    - ALPHA * d_n0.signal
reference_corr = d_nv.reference - ALPHA * d_n0.reference

# Avoid division by zero
with np.errstate(divide='ignore', invalid='ignore'):
    contrast_corr = np.where(
        np.abs(reference_corr) > 0,
        (reference_corr - signal_corr) / reference_corr * 100,
        0.0
    )

freqs_MHz = d_nv.frequencies

print("Noise correction applied.")
print(f"  Original noise std (contrast): {np.std(d_nv.contrast_percent):.4f} %")
print(f"  Corrected noise std (contrast): {np.std(contrast_corr):.4f} %")

In [ ]:
# ── Plot: raw vs noise-corrected ODMR ─────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# Signal & reference (raw NV channel)
ax = axes[0]
ax.plot(freqs_MHz, d_nv.signal,    label=f'MW on  (Ch{NV_CHANNEL} raw)',  color='steelblue')
ax.plot(freqs_MHz, d_nv.reference, label=f'MW off (Ch{NV_CHANNEL} raw)',  color='darkorange')
ax.plot(freqs_MHz, signal_corr,    label='MW on  (corrected)', color='steelblue',   linestyle='--', alpha=0.7)
ax.plot(freqs_MHz, reference_corr, label='MW off (corrected)', color='darkorange',  linestyle='--', alpha=0.7)
ax.set_ylabel('PL Intensity (ADC units)')
ax.set_title('ODMR: Signal & Reference — raw vs noise-corrected')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Contrast comparison
ax = axes[1]
ax.plot(freqs_MHz, d_nv.contrast_percent, color='steelblue', linewidth=1.2, alpha=0.8,
        label='Raw contrast (Ch1 only)')
ax.plot(freqs_MHz, contrast_corr,         color='green',     linewidth=1.5,
        label='Noise-corrected contrast')
ax.axhline(0, color='k', linewidth=0.5, linestyle='--')
ax.set_ylabel('Contrast (%)')
ax.set_title('ODMR Contrast — raw vs noise-corrected')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Noise reference channel contrast (shows what was subtracted)
ax = axes[2]
ax.plot(freqs_MHz, d_n0.contrast_percent, color='darkorange', linewidth=1.2, alpha=0.8,
        label=f'Noise ref contrast (Ch{NOISE_CHANNEL} — laser noise only)')
ax.axhline(0, color='k', linewidth=0.5, linestyle='--')
ax.set_ylabel('Contrast (%)')
ax.set_xlabel('Frequency (MHz)')
ax.set_title(f'Noise reference channel (Ch{NOISE_CHANNEL}) — ideally flat; non-zero means laser noise has spectral structure')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Find ODMR dips in the noise-corrected spectrum ─────────────────────────
def find_odmr_dips(frequencies, contrast_percent, n_peaks=4, prominence_factor=1.5):
    """Find ODMR dip positions. Dips are minima in contrast_percent."""
    inverted = -contrast_percent
    threshold = np.std(inverted) * prominence_factor
    peak_idx, _ = scipy_find_peaks(
        inverted,
        height=threshold,
        distance=len(frequencies) // (n_peaks * 3)
    )
    sort_order = np.argsort(frequencies[peak_idx])
    return peak_idx[sort_order]

dip_idx = find_odmr_dips(freqs_MHz, contrast_corr, n_peaks=4)

plt.figure(figsize=(12, 4))
plt.plot(freqs_MHz, d_nv.contrast_percent, color='steelblue', linewidth=1.2, alpha=0.6, label='Raw')
plt.plot(freqs_MHz, contrast_corr,         color='green',     linewidth=1.5,            label='Noise-corrected')
plt.plot(freqs_MHz[dip_idx], contrast_corr[dip_idx], 'rv', markersize=10, label='Dips (corrected)')

for idx in dip_idx:
    plt.annotate(
        f"{freqs_MHz[idx]:.1f} MHz",
        xy=(freqs_MHz[idx], contrast_corr[idx]),
        xytext=(0, -18), textcoords='offset points',
        ha='center', fontsize=9, color='darkred'
    )

plt.axhline(0, color='k', linewidth=0.5, linestyle='--')
plt.ylabel('Contrast (%)')
plt.xlabel('Frequency (MHz)')
plt.title(f'ODMR Dips (noise-corrected) — {len(dip_idx)} found')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Detected ODMR dips (noise-corrected):")
for i, idx in enumerate(dip_idx):
    print(f"  Dip {i+1}: {freqs_MHz[idx]:.2f} MHz  (contrast: {contrast_corr[idx]:.2f} %)")

In [ ]:
# ===== EDIT THIS: Set your chosen resonance frequency =====
# Pick one well-resolved dip from the noise-corrected spectrum above.

RESONANCE_FREQ_MHZ = 2870.0   # <-- update from noise-corrected ODMR spectrum

print(f"Selected resonance frequency: {RESONANCE_FREQ_MHZ} MHz")

In [ ]:
# Live noise-corrected ODMR
# Watch peaks shift as you adjust the bias magnet.
# Press Stop (■) to exit.

def get_corrected_odmr():
    d_nv_ = prog_odmr_nv.acquire(progress=False)
    d_n0_ = prog_odmr_n0.acquire(progress=False)
    sig_c = d_nv_.signal    - ALPHA * d_n0_.signal
    ref_c = d_nv_.reference - ALPHA * d_n0_.reference
    with np.errstate(divide='ignore', invalid='ignore'):
        contrast_c = np.where(np.abs(ref_c) > 0,
                              (ref_c - sig_c) / ref_c * 100, 0.0)
    return d_nv_.frequencies, contrast_c

# Uncomment to start:
# qd.live_plot(get_corrected_odmr)

---
## Section 6: Noise-Corrected Readout Window Calibration

The readout window measurement is run on **both** ADC channels.  
The corrected difference signal (used for exponential decay fitting):
$$\delta_{\text{corr}}(t) = [\text{ref}_{\text{ch1}}(t) - \text{sig}_{\text{ch1}}(t)] - \alpha\,[\text{ref}_{\text{ch0}}(t) - \text{sig}_{\text{ch0}}(t)]$$

This suppresses laser-intensity fluctuations from the time constant fit, giving a cleaner estimate of the repumping time constant τ.

In [ ]:
# ── Shared readout window parameters ──────────────────────────────────────
def make_rw_config(adc_ch):
    cfg = make_config(adc_ch)
    cfg.mw_gain                    = 30000
    cfg.mw_fMHz                    = RESONANCE_FREQ_MHZ
    cfg.mw_nqz                     = 1
    cfg.mw_pi2_tns                 = 100        # Rough estimate; calibrate with Rabi in NB 2
    cfg.relax_delay_tus            = 1
    cfg.readout_length_treg        = 1020
    cfg.laser_initialize_tus       = 15
    cfg.mw_readout_delay_treg      = 35
    cfg.laser_readout_offset_treg  = 0
    cfg.soft_avgs                  = 200
    cfg.pre_init                   = True
    return cfg

print("Acquiring readout window on NV channel...")
data_on_nv, data_off_nv, prog_rw_nv = qd.get_readout_window(make_rw_config(NV_CHANNEL),    n_time_bins=3)
print("Acquiring readout window on noise channel...")
data_on_n0, data_off_n0, prog_rw_n0 = qd.get_readout_window(make_rw_config(NOISE_CHANNEL), n_time_bins=3)
print("Done.")

# Noise-corrected on/off traces
data_on_corr  = data_on_nv  - ALPHA * data_on_n0
data_off_corr = data_off_nv - ALPHA * data_off_n0

In [ ]:
t_rw = np.arange(len(data_off_nv)) * qd.min_time_tus   # μs

norm_ref = np.max(np.abs(data_off_nv))

fig, axes = plt.subplots(3, 1, figsize=(12, 11), sharex=True)

# Raw NV channel
ax = axes[0]
ax.plot(t_rw, data_on_nv  / norm_ref, label='MW π/2 on  (Ch1 raw)',  color='steelblue',  linewidth=1.5)
ax.plot(t_rw, data_off_nv / norm_ref, label='MW off     (Ch1 raw)',  color='darkorange', linewidth=1.5)
ax.set_ylabel('PL (normalised)')
ax.set_title('Readout Window — Raw NV Channel (Ch1)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Noise reference channel
norm_n0 = np.max(np.abs(data_off_n0)) if np.max(np.abs(data_off_n0)) > 0 else 1
ax = axes[1]
ax.plot(t_rw, data_on_n0  / norm_n0, label='MW π/2 on  (Ch0 noise ref)', color='steelblue',  linewidth=1, linestyle='--')
ax.plot(t_rw, data_off_n0 / norm_n0, label='MW off     (Ch0 noise ref)', color='darkorange', linewidth=1, linestyle='--')
ax.set_ylabel('Noise ref (normalised)')
ax.set_title(f'Noise Reference Channel (Ch{NOISE_CHANNEL}) — should be flat (MW-independent)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Corrected difference
diff_raw  = data_off_nv  - data_on_nv
diff_corr = data_off_corr - data_on_corr
ax = axes[2]
ax.plot(t_rw, diff_raw,  color='steelblue', linewidth=1.5, alpha=0.7, label='Difference — raw (ch1)')
ax.plot(t_rw, diff_corr, color='green',     linewidth=1.5,             label='Difference — noise-corrected')
ax.axhline(0, color='k', linewidth=0.5, linestyle='--')
ax.set_ylabel('Difference (ref − sig)')
ax.set_xlabel('Time (μs)')
ax.set_title('Readout Window Difference — raw vs noise-corrected')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Signal onset from corrected trace
norm_off_corr = data_off_corr / (np.max(np.abs(data_off_corr)) or 1)
onset_cands = np.where(norm_off_corr > 0.8)[0]
if len(onset_cands) > 0:
    print(f"Signal onset (corrected): ~{t_rw[onset_cands[0]]:.2f} μs")
    print(f"Suggested laser_readout_offset: ~{onset_cands[0]} reg units")
else:
    print("Signal appears saturated from t=0 — offset likely 0 (expected for manual laser).")

In [ ]:
# ── Fit exponential decay to noise-corrected difference ───────────────────
skip = 10
diff_fit = diff_corr[skip:]
t_fit    = t_rw[skip:]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t_rw,   diff_raw,  color='steelblue', linewidth=1.2, alpha=0.6, label='Raw difference')
ax.plot(t_fit,  diff_fit,  color='green',     linewidth=1.5,             label='Corrected (fit region)')

try:
    popt, _ = curve_fit(
        qd.exponential_decay, t_fit, diff_fit,
        p0=[np.max(diff_fit), 0.8, 0],
        maxfev=5000
    )
    tau = popt[1]
    ax.plot(t_fit, qd.exponential_decay(t_fit, *popt), 'r--', linewidth=2,
            label=f'Fit  τ = {tau:.2f} μs  (noise-corrected)')
    ax.legend()
    print("Exponential decay fit (noise-corrected):")
    print(f"  Time constant τ = {tau:.2f} μs")
    print()
    print("Recommended parameters for pulsed measurements (Notebook 2):")
    print(f"  readout_integration_tus  ≈ {tau:.1f} μs   (1 × τ)")
    print(f"  laser_initialize_tus     ≈ {5*tau:.1f} μs   (5 × τ)")
except RuntimeError:
    print("Curve fit failed. Try adjusting `skip` or p0 initial guesses.")
    tau = None

ax.axhline(0, color='k', linewidth=0.5, linestyle='--')
ax.set_ylabel('Difference (ref − sig)')
ax.set_xlabel('Time (μs)')
ax.set_title('Noise-Corrected Readout Window Decay Fit')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ===== EDIT THESE: Save calibrated readout parameters =====
READOUT_OFFSET_TREG      = 0      # <-- From onset analysis
READOUT_INTEGRATION_TUS  = 2.0   # <-- From noise-corrected exponential fit: ~1 × τ
LASER_ON_TUS             = 15.0  # <-- From fit: ≥ 5 × τ
MW_READOUT_DELAY_TREG    = 35    # <-- Keep unless timing issues arise

print("Calibrated readout parameters (noise-corrected):")
print(f"  READOUT_OFFSET_TREG:     {READOUT_OFFSET_TREG}")
print(f"  READOUT_INTEGRATION_TUS: {READOUT_INTEGRATION_TUS} μs")
print(f"  LASER_ON_TUS:            {LASER_ON_TUS} μs")
print(f"  MW_READOUT_DELAY_TREG:   {MW_READOUT_DELAY_TREG}")
print(f"  ALPHA:                   {ALPHA:.4f}  (noise gain factor — bring to NB 2)")
print()
print("Copy these values into the Master Configuration cell at the top of Notebook 2.")

---
## Summary

**What was accomplished in this notebook:**
- ✅ Connected to RFSoC and verified QICK-DAWG
- ✅ Confirmed both photodiode channels (Ch1 NV PL, Ch0 noise reference)
- ✅ Calibrated laser noise gain factor **α** from correlated noise floor
- ✅ Acquired noise-corrected PL, ODMR spectrum, and readout window

**Proceed to `02_rabi_vector_magnetometry.ipynb`** with these values:

```python
RESONANCE_FREQ_MHZ       = {RESONANCE_FREQ_MHZ}     # Section 5
READOUT_OFFSET_TREG      = {READOUT_OFFSET_TREG}     # Section 6
READOUT_INTEGRATION_TUS  = {READOUT_INTEGRATION_TUS} # Section 6
LASER_ON_TUS             = {LASER_ON_TUS}            # Section 6
MW_READOUT_DELAY_TREG    = {MW_READOUT_DELAY_TREG}   # Section 6
ALPHA                    = {ALPHA:.4f}               # noise gain factor
```

### Noise cancellation tips

| Issue | Likely cause | Fix |
|-------|-------------|-----|
| Low correlation `r` | Reference detector not on same laser beam | Check Ch0 optical path |
| α < 0 | Channels inverted | Swap photodiode polarity on Ch0 |
| Noise reduction < 20 % | Electronic noise dominates over laser noise | Improve shielding; check amplifier |
| Corrected ODMR noisier than raw | α mis-calibrated | Re-run Section 3 with more samples |